In [1]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics.pairwise import paired_cosine_distances

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

model_name = "sentence-transformers/paraphrase-distilroberta-base-v1"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


{'model_name': 'sentence-transformers/paraphrase-distilroberta-base-v1', 'dataset': 'glue/stsb', 'split': 'validation', 'device': 'mps', 'batch_size': 128, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy().reset_index(drop=True)

n = len(df)
lower_idx = int(n * 0.25)
upper_idx = int(n * 0.75)
df_slice = df.iloc[lower_idx:upper_idx].reset_index(drop=True)

print({
    "num_examples_full": n,
    "slice_start": lower_idx,
    "slice_end": upper_idx,
    "num_examples_slice": len(df_slice),
    "columns": df_slice.columns.tolist(),
})
print(df_slice.head())


{'num_examples_full': 1500, 'slice_start': 375, 'slice_end': 1125, 'num_examples_slice': 750, 'columns': ['sentence1', 'sentence2', 'label']}
                                           sentence1  \
0          A man eating an apple, sitting in public.   
1       two young girls hug each other in the grass.   
2  Two white dogs are walking through deep white ...   
3  Four girls in swimsuits are playing volleyball...   
4                     Kids playing ball in the park.   

                                           sentence2  label  
0      A man sits in a public place eating an apple.    5.0  
1  two young girls holding each other on the gras...    4.6  
2  Two white dogs walk through a huge bank of mou...    4.6  
3  Four women in bikinis are playing volleyball o...    4.8  
4                     Two dogs playing by the shore.    0.6  


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

In [ ]:
sentences1 = df_slice["sentence1"].tolist()
sentences2 = df_slice["sentence2"].tolist()
labels = df_slice["label"].to_numpy(dtype=np.float32)

emb1_tokens = model.encode(
    sentences1,
    batch_size=batch_size,
    output_value="token_embeddings",
    convert_to_numpy=False,
    show_progress_bar=False,
)

emb2_tokens = model.encode(
    sentences2,
    batch_size=batch_size,
    output_value="token_embeddings",
    convert_to_numpy=False,
    show_progress_bar=False,
)

emb1 = np.stack([token_tensor[0].detach().cpu().numpy() for token_tensor in emb1_tokens]).astype(np.float32)
emb2 = np.stack([token_tensor[0].detach().cpu().numpy() for token_tensor in emb2_tokens]).astype(np.float32)

embedding_dim = int(emb1.shape[1])
cosine_similarity_raw = 1.0 - paired_cosine_distances(emb1, emb2)
predicted_score_0_5 = 2.5 * (cosine_similarity_raw + 1.0)

print({
    "embedding_dim": embedding_dim,
    "emb1_shape": emb1.shape,
    "emb2_shape": emb2.shape,
})


In [ ]:
pearson_raw = pearsonr(cosine_similarity_raw, labels).statistic
spearman_raw = spearmanr(cosine_similarity_raw, labels).statistic
pearson_rescaled = pearsonr(predicted_score_0_5, labels).statistic
spearman_rescaled = spearmanr(predicted_score_0_5, labels).statistic

results_df = df_slice.copy()
results_df["cosine_similarity_raw"] = cosine_similarity_raw
results_df["predicted_score_0_5"] = predicted_score_0_5

top_examples = results_df.sort_values("cosine_similarity_raw", ascending=False).head(5).reset_index(drop=True)
bottom_examples = results_df.sort_values("cosine_similarity_raw", ascending=True).head(5).reset_index(drop=True)

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity_raw", "predicted_score_0_5"]].head(10))
print("top_scoring_examples")
print(top_examples[["sentence1", "sentence2", "label", "cosine_similarity_raw", "predicted_score_0_5"]])
print("bottom_scoring_examples")
print(bottom_examples[["sentence1", "sentence2", "label", "cosine_similarity_raw", "predicted_score_0_5"]])


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"slice_bounds: [{lower_idx}, {upper_idx})")
print(f"num_examples_full: {n}")
print(f"num_examples_slice: {len(df_slice)}")
print(f"embedding_dimensionality: {embedding_dim}")
print(f"pooling_strategy: first_token_cls_style_from_token_embeddings")
print(f"pearson_raw_cosine: {pearson_raw:.6f}")
print(f"spearman_raw_cosine: {spearman_raw:.6f}")
print(f"pearson_rescaled_0_5: {pearson_rescaled:.6f}")
print(f"spearman_rescaled_0_5: {spearman_rescaled:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
